# B02 · Classical reference: SAR texture features + random forest (development events, CPU only)

Do frozen foundation-model features beat a **classical texture pipeline** on the same pixels? For every building:
intensity statistics and grey-level co-occurrence (GLCM) texture of its footprint, and, as a second setting, the
same features of a 32 m square around it (context). A random forest (300 trees, fixed settings in
`src/damagesar_bench/baselines/texture.py`, never tuned on a held-out event) is trained per held-out development
event on every other development and secondary event, as in building-level SAR damage work with texture
features and a random forest (Macchiarulo et al., 2025). **Exploratory reference, not a contribution.**

Compared with the frozen probes of p03 (DINOv3 ViT-B, ImageNet ViT-B, image-free control) and p04 (satellite
DINOv3 ViT-L) on the same events, with the same event-level statistics.

* **Before running:** Runtime → Change runtime type → **CPU** is enough (no GPU needed). Secret: `GH_TOKEN`.
  Needs p01 (building table), p03 and p04 (their run folders).
* **Time:** about 25–35 minutes (features ~10 min, 2 settings × 5 forests). No compute units on a CPU runtime.
* **Outputs:** `texture_features.parquet`, `fold_scores.parquet`, `event_metrics.csv`, `comparisons.csv` in this
  run's folder.

In [ ]:
NOTEBOOK_NAME = "b02_texture_baseline"

# Drive layout (FYP Google account). Keep in sync with src/damagesar_bench/drive.py
import os, datetime, re
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/DamageSAR-Bench"
FOLDERS = ["00_admin", "01_data/raw/bright", "01_data/raw/instance_labels", "01_data/external",
           "01_data/derived", "02_models/checkpoints", "02_models/features", "03_notebooks/pipeline",
           "03_notebooks/checks", "03_notebooks/scratch", "04_outputs/runs", "04_outputs/reports"]
for f in FOLDERS:
    os.makedirs(f"{PROJECT}/{f}", exist_ok=True)
def run_dir(name):
    slug = re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")
    p = f"{PROJECT}/04_outputs/runs/{datetime.date.today().isoformat()}_{slug}"
    os.makedirs(p, exist_ok=True)
    return p
RUN = run_dir(NOTEBOOK_NAME)
print("Drive layout ready under", PROJECT)
print("This run saves to:", RUN)

In [ ]:
# Install this repo's package (private repo: needs GH_TOKEN in Colab Secrets, read access to damagesar-bench)
from google.colab import userdata
GH = userdata.get("GH_TOKEN")
!git clone -q https://{GH}@github.com/ThinukaLehan/damagesar-bench.git /content/damagesar-bench 2>/dev/null || (cd /content/damagesar-bench && git pull -q)
!pip install -q -e /content/damagesar-bench
COMMIT = !git -C /content/damagesar-bench rev-parse --short HEAD
COMMIT = COMMIT[0]
# A running Colab kernel does not see an editable install until a restart: put src/ on the path.
import sys, importlib
if "/content/damagesar-bench/src" not in sys.path:
    sys.path.insert(0, "/content/damagesar-bench/src")
importlib.invalidate_caches()
import damagesar_bench
print("damagesar-bench at commit", COMMIT, "->", damagesar_bench.__file__)

In [ ]:
# Inputs: the p03 and p04 run folders
import glob, os
P03 = sorted(glob.glob(f"{PROJECT}/04_outputs/runs/*_p03-controls-and-imagenet-v0/bench_run"))
P04 = sorted(glob.glob(f"{PROJECT}/04_outputs/runs/*_p04-more-encoders-and-thresholds/bench_run"))
assert P03 and P04, "missing the p03 or p04 run folder: run them first"
P03, P04 = P03[-1], P04[-1]
print("p03:", P03, "| p04:", P04, "| CPUs:", os.cpu_count())

In [ ]:
# BRIGHT post-event SAR only (~2 GB): labels come from the building table
import zipfile
from huggingface_hub import hf_hub_download
ROOT = "/content/bright"
post = f"{ROOT}/post-event"
if not (os.path.isdir(post) and len(os.listdir(post)) > 4000):
    p = hf_hub_download("Kullervo/BRIGHT", "post-event.zip", repo_type="dataset", local_dir="/content/zips")
    with zipfile.ZipFile(p) as z:
        z.extractall(ROOT)
print(len(os.listdir(post)), "post-event tiles")

In [ ]:
# Building table from p01: development + secondary events only (confirmatory stay locked)
import pandas as pd
from damagesar_bench.config import load_events
events = load_events("/content/damagesar-bench/configs/events.yaml")
B = pd.read_parquet(f"{PROJECT}/01_data/derived/buildings.parquet")
B["damaged"] = B["damaged"].astype("boolean")
use = B[B["role"].isin(["development", "secondary"])].copy()
del B
print(use[use["excluded_reason"].isna()].groupby("event").size().to_string())

In [ ]:
# 1. Texture features for every kept building (footprint and 32 m context square)
import time
from damagesar_bench.baselines.texture import leave_one_event_out, texture_table
from damagesar_bench.encoders.extract import read_bright_tile
t0 = time.time()
T = texture_table(use, lambda t: read_bright_tile(ROOT, t), log=print)
T.to_parquet(f"{RUN}/texture_features.parquet", index=False)
print(len(T), "buildings,", round(time.time() - t0), "s")

In [ ]:
# 2. One random forest per held-out development event, for each setting
dev = events.keys_with_role("development")
FS = pd.concat([leave_one_event_out(T, dev, p, log=print) for p in ("footprint", "footprint_ctx32m")],
               ignore_index=True)
FS.to_parquet(f"{RUN}/fold_scores.parquet", index=False)
pd.DataFrame([{"seconds": round(time.time() - t0), "buildings": len(T), "cpus": os.cpu_count()}]).to_csv(
    f"{RUN}/timing.csv", index=False)

In [ ]:
# 3. Per-event AUROC and event-level comparisons with the frozen probes (same events, same statistics)
import json
from damagesar_bench.stats.compare import compare, event_metrics
M = event_metrics(FS, events, n_boot=2000, seed=0)
M.to_csv(f"{RUN}/event_metrics.csv", index=False)
ALL = pd.concat([M, pd.read_csv(f"{P03}/event_metrics.csv"), pd.read_csv(f"{P04}/event_metrics.csv")],
                ignore_index=True).drop_duplicates(["encoder", "pooling", "event"])
TX, TXC = ("texture_rf", "footprint"), ("texture_rf", "footprint_ctx32m")
SAT, IMN = ("dinov3_vitl16_sat@native", "outline"), ("imagenet_vitb16@native", "outline")
D3, FREE = ("dinov3_vitb16@native", "outline"), ("imagefree", "geometry")
rows = []
for label, a, b in [("texture vs image-free", TX, FREE), ("texture context vs footprint", TXC, TX),
                    ("DINOv3 ViT-B probe vs texture", D3, TX), ("ImageNet probe vs texture", IMN, TX),
                    ("satellite ViT-L probe vs texture", SAT, TX)]:
    c = compare(ALL, a, b, events=dev)
    rows.append({"label": label, "estimate": c.interval.estimate, "lower": c.interval.lower,
                 "upper": c.interval.upper, "verdict": c.verdict.meaning, "sign_flip_p": c.sign_flip.p_value,
                 "gains": json.dumps({k: round(float(v), 6) for k, v in c.gains.items()})})
C = pd.DataFrame(rows); C.to_csv(f"{RUN}/comparisons.csv", index=False)
keep = [TX, TXC, SAT, IMN, D3, FREE]
pv = ALL[[k in keep for k in zip(ALL["encoder"], ALL["pooling"])]].pivot_table(
    index=["encoder", "pooling"], columns="event", values="auroc")
pv["mean"] = pv.mean(axis=1)
print(pv.round(3).to_string()); print(C.drop(columns="gains").round(4).to_string(index=False))

In [ ]:
# File the executed notebook (with outputs) into this run's folder and 03_notebooks/pipeline/
import json
from google.colab import _message
# Ask the browser for the notebook; it can time out (busy tab, large outputs). Results above are already saved.
nb = None
for attempt in range(3):
    try:
        reply = _message.blocking_request("get_ipynb", timeout_sec=120)
        nb = (reply or {}).get("ipynb")
    except Exception as ex:
        print("attempt", attempt + 1, "failed:", ex)
    if nb:
        break
if not nb:
    print("Could not copy the executed notebook automatically (the results above are already saved).")
    print("Use File -> Save a copy in Drive, then move the copy into", RUN)
else:
    stamp = datetime.date.today().isoformat()
    for target in (f"{RUN}/{NOTEBOOK_NAME}_executed.ipynb",
                   f"{PROJECT}/03_notebooks/pipeline/{stamp}_{NOTEBOOK_NAME}.ipynb"):
        with open(target, "w") as fh:
            json.dump(nb, fh)
        print("saved", target)
    print("run folder:", sorted(os.listdir(RUN)))